# Session 2.8: Danke! Nun geht's ans Projekt

## Projekt: GenAI/Conversational AI Formular-Ausfüllassistent PoC (Proof of Concept)

### Beschreibung
- Oft gibt es Formulare oder Fragebögen, die sehr spezifische Fragen beinhalten und mit denen sich User:innen überfordert fühlen
- Gerne hätten sie jemanden, um näher Fragen stellen zu können oder Vorschläge/Beispiele geben zu können
- Ein GenAI-Assistent hat das Potenzial, hier eine automatisierte beliebig vervielfältigbare Lösung darzustellen
- Conversational AI einer der Use Cases mit meistem Potenzial
- PoC = Minimal; zeigen, dass etwas geht oder nicht geht oder vielleicht gehen könnte

### Kurze Demo: Formfilling Agent

### Use Cases
- Gerätesupport-Assistent
- Mitarbeiter-Richtlinien-Assistent
- Kontoeröffnungs-Assitent
- Versicherungsberatungs-Assistent
- Reiseplanungs-Assistent
- Bauantrags-Assistent
- Studienbewerbungs-Assistent
- Fördermittel-Antragsassistent
- Datenschutz-Compliance-Assistent
- Steuerererklärungs-Assistent
- Patent-Anmeldeassistent
- Lebensmittel-Beratungsassistent
- ...

### Mögliche Tool-Vorschläge
- RAG
- Wetter-API
- Währungsrechner-API
- Internetsuche
- Flug-/Zugverbindungs-API
- Kalendermanagement
- Openstreetmap
- Prozentrechner
- OpenFoodFacts (Lebensmittelinformationen)
- ...

### Punkteverteilung Projektarbeit: Lieferobjekte
- 08P | 20 Seiten PDF (ca. 12k Wörter) zu einem Thema oder mehreren Themen einer Kategorie
- 10P | Document Processing - diese 20 Seiten PDF geparst und gechunkt
- 12P | Vektorisierung und Storage - diese 20 Seiten PDF embedded und indexiert in einer lokalen VektorDB
- 12P | Agent-Tool für RAG - Retrieval aus der Vektor-DB durch Embedding der Query mit Embedding-Model
- 10P | JSON Schema und Pydantic Models, mit mindestens 3 "Klassen" und je Klasse mindestens 3 "Felder", die von User:in gewünscht sind zu befüllen
    - Beispiel: class Geschäftsanforderung(BaseModel):
        - Problemstellung: Optional[str] = Field(None, description="Welche Problemstellung soll gelöst werden?")
        - Ziel: Optional[str] = Field(None, description="Welches Ziel soll erreicht werden?")
        - Ergebnis: Optional[str] = Field(None, description="Welches Ergebnis soll erzielt werden?")
        - FachlicheAnforderungen: Optional[str] = Field(None, description="Welche fachlichen Anforderungen sollen erfüllt werden?")
        - NichtFunktionaleAnforderungen: Optional[str] = Field(None, description="Welche nicht funktionalen Anforderungen gibt es?")
- 17P | Weitere nötige Tools, nötige Agenten, Kontextmanagement -> Anwendung von Agent Patterns
    - Beispiele Tools
        - extract_form_fields
        - update_form
        - get_form_summary
        - save_form
        - ...
    - Beispiele Agenten
        - Dynamic Form Agent
        - Form Field Extractor
- 10P | UseCase-Beschreibung und Mermaid Diagramm (Sequence-Diagramm oder wenn gewünscht ein anderes)
- 10P | Anwendungsfunktionalität: lässt sich starten, lässt sich von Beginn bis Ende durchführen mit einem JSON Endergebnis (Chat als Loop)
- 06P | Stats und Tracking (Tokenverbrauch, Items und deren Abfolgen in RunResult) bewusst und dokumentiert
- 05P | mind. 3 erfolgreiche Fälle, mind. 3 weniger erfolgreiche Fälle - genug Erfahrungen teilen
- GESAMT 100P -> macht 50% der Gesamtnote aus



### Präsentation
- Infos
    - 10 Minuten pro Gruppe
    - Ihr könnt entscheiden, wer präsentiert - Punkte bekommt die ganze Gruppe
    - Visuelle Unterstützung - wie, entscheidet ihr (Powerpoint, Dokument, ...)
    - Es muss ein Live Demo Element dabei sein
    - Abgabe: Visuelle Unterstützung auf Moodle unter "Präsentation" hochladen
- Punkteverteilung
    - 10P | Einhaltung 10 Minuten +/- 2 Minuten
    - 15P | Beschreibung des Use-Cases
    - 20P | Soll eine Live-Demo enthalten
    - 10P | Soll ein Diagramm enthalten (Sequence, Prozess-Flow, ...)
    - 25P | Benützte Komponenten, Parameter/Stats und Erfahrungen bei allen Schritten in der RAG-Pipeline 
        - [Parsing -> Chunking -> Embedding -> Indexing -> Retrieval (Abruf) -> (Reranking) -> Augmented (erweiterte) Generation]
        - Welches Embedding-Model (welche Dimensionen), welches Generation-Model? Verschiedene ausprobiert?
        - Welche Agent Patterns wurden angewandt?
    - 20P | Grober Code-Walkthrough
    - GESAMT 100P -> macht 20% der Gesamtnote aus

### Weiteres

- Pydantic JSON Parsing: https://docs.pydantic.dev/latest/concepts/json/

In [ ]:
# -*- coding: utf-8 -*-
# =============================================================================
# Projektarbeit: GenAI/Conversational AI Bauantrags-Assistent (Mit Ollama LLM)
# =============================================================================
# Ziel: Erfüllung der Projektanforderungen mit Integration eines lokalen LLM
#       via Ollama für RAG-Antwortgenerierung und Umformulierung.
# =============================================================================

# =============================================================================
# 0. Setup und Imports
# =============================================================================
import os
import json
from typing import Optional, List, Dict, Any, Union
import datetime

# Pydantic für Datenstruktur-Definition
from pydantic import BaseModel, Field, ValidationError

# LangChain Komponenten
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
# Importiere Ollama LLM
from langchain_community.llms import Ollama

print("Bibliotheken importiert.")

# =============================================================================
# A. Konfiguration
# =============================================================================
# --- PDF und RAG Konfiguration ---
pdf_file_path = "assets/NÖ  BO 2014, Fassung gekürzt.pdf"
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
chunk_size = 500
chunk_overlap = 50
k_retriever = 3

# Beispiele: "llama3:8b", "mistral:7b", "phi3:medium"
OLLAMA_MODEL_NAME = "llama3.2:latest"

# =============================================================================
# B. LLM Initialisierung (Globales Objekt)
# =============================================================================
llm = None # Initialisieren als None
try:
    print(f"Versuche, Ollama LLM zu initialisieren (Modell: {OLLAMA_MODEL_NAME})...")
    # Hier wird das globale llm-Objekt erstellt, wenn Ollama erreichbar ist
    llm = Ollama(model=OLLAMA_MODEL_NAME)
    # Teste die Verbindung kurz
    llm.invoke("Hallo!") # Einfacher Aufruf zum Testen
    print(f"INFO: Ollama LLM '{OLLAMA_MODEL_NAME}' erfolgreich initialisiert und verbunden.")
    LLM_AVAILABLE = True
except Exception as e:
    print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    print(f"FEHLER: Konnte Ollama LLM nicht initialisieren oder verbinden.")
    print(f"Fehlermeldung: {e}")
    print("Mögliche Ursachen:")
    print("  - Ollama Dienst läuft nicht auf deinem System.")
    print(f"  - Das Modell '{OLLAMA_MODEL_NAME}' ist nicht in Ollama heruntergeladen (`ollama pull {OLLAMA_MODEL_NAME}`).")
    print("  - Netzwerkprobleme, falls Ollama auf einem anderen Host läuft.")
    print("-> LLM-Funktionen (RAG-Antworten, Umformulierung) werden nur SIMULIERT.")
    print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    llm = None # Sicherstellen, dass llm None ist, wenn Initialisierung fehlschlägt
    LLM_AVAILABLE = False


# =============================================================================
# 1. JSON Schema und Pydantic Models (10P)
# =============================================================================
# ... (Pydantic Models Definition bleibt unverändert) ...
class ProjektDetails(BaseModel):
    bauprojekt_art: Optional[str] = Field(None, description="Art des Bauprojekts (z.B. Neubau Einfamilienhaus, Anbau Garage, Dachgeschossausbau). Was genau soll gebaut oder verändert werden?")
    geplante_nutzung: Optional[str] = Field(None, description="Wie soll das Bauprojekt genutzt werden? (z.B. Wohnen, Gewerbe, Lager)")
    gebaeudehoehe_geplant: Optional[float] = Field(None, description="Wie hoch soll das Gebäude maximal werden (in Metern)?")
    barrierefreiheit_geplant: Optional[bool] = Field(None, description="Ist eine barrierefreie Ausführung geplant oder vorgeschrieben? (ja/nein)")

class GrundstueckInformationen(BaseModel):
    adresse_strasse_nr: Optional[str] = Field(None, description="Straße und Hausnummer des Baugrundstücks.")
    adresse_plz_ort: Optional[str] = Field(None, description="PLZ und Ort des Baugrundstücks.")
    grundstueckflaeche_qm: Optional[int] = Field(None, description="Gesamtfläche des Grundstücks in Quadratmetern.")
    liegt_in_schutzgebiet: Optional[bool] = Field(None, description="Liegt das Grundstück in einem bekannten Schutzgebiet (z.B. Wasserschutz, Naturschutz)? (ja/nein)")

class AntragstellerKontakt(BaseModel):
    name_vorname: Optional[str] = Field(None, description="Vollständiger Name des Antragstellers.")
    anschrift: Optional[str] = Field(None, description="Anschrift des Antragstellers (falls abweichend vom Bauort).")
    telefonnummer: Optional[str] = Field(None, description="Telefonnummer für Rückfragen.")
    email: Optional[str] = Field(None, description="E-Mail-Adresse für Rückfragen.")
    ist_grundstueck_eigentuemer: Optional[bool] = Field(None, description="Ist der Antragsteller auch der Eigentümer des Baugrundstücks? (ja/nein)")

class Bauantrag(BaseModel):
    projekt: ProjektDetails = Field(default_factory=ProjektDetails, description="Details zum geplanten Bauprojekt.")
    grundstueck: GrundstueckInformationen = Field(default_factory=GrundstueckInformationen, description="Informationen zum Baugrundstück.")
    antragsteller: AntragstellerKontakt = Field(default_factory=AntragstellerKontakt, description="Kontaktdaten des Antragstellers.")
    architekt_beauftragt: Optional[bool] = Field(None, description="Wurde ein Architekt oder Planer mit der Planung beauftragt? (ja/nein)")
    geplanter_baubeginn: Optional[str] = Field(None, description="Ungefähres Datum des geplanten Baubeginns (z.B. YYYY-MM).")
    energieeffizienzklasse_angestrebt: Optional[str] = Field(None, description="Welche Energieeffizienzklasse wird angestrebt (z.B. A+, A, B)?")

    @classmethod
    def get_all_field_descriptions(cls) -> Dict[str, str]:
        descriptions = {}
        schema = cls.model_json_schema()
        def extract_descriptions(sub_schema, prefix=""):
            if 'properties' in sub_schema:
                for key, value in sub_schema['properties'].items():
                    field_path = f"{prefix}{key}"
                    if 'description' in value: descriptions[field_path] = value['description']
                    if '$ref' in value:
                         ref_name = value['$ref'].split('/')[-1]
                         if ref_name in schema.get('$defs', {}): extract_descriptions(schema['$defs'][ref_name], prefix=f"{field_path}.")
                    elif value.get('type') == 'object': extract_descriptions(value, prefix=f"{field_path}.")
        extract_descriptions(schema)
        return descriptions

print("Pydantic Models definiert.")

# =============================================================================
# 2. Document Processing (10P) & Vektorisierung/Storage
# =============================================================================
# ... (PDF Verarbeitung / Vektorisierung bleibt unverändert) ...
retriever = None
pdf_processed_successfully = False
if not os.path.exists(pdf_file_path): print(f"FEHLER: PDF-Datei nicht gefunden: {pdf_file_path}")
else:
    print(f"Verarbeite PDF: {pdf_file_path}")
    try:
        loader = PyMuPDFLoader(pdf_file_path); pages = loader.load(); print(f"-> PDF geladen ({len(pages)} Seiten).")
        if pages:
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap); chunks = text_splitter.split_documents(pages); print(f"-> Dokument in {len(chunks)} Chunks aufgeteilt.")
            if chunks:
                print(f"-> Lade Embedding-Modell: {embedding_model_name}..."); embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name); print("-> Embedding-Modell geladen.")
                print("-> Erstelle FAISS Vektordatenbank..."); vectorstore = FAISS.from_documents(chunks, embeddings); retriever = vectorstore.as_retriever(search_kwargs={"k": k_retriever}); print(f"-> Vektordatenbank und Retriever erstellt (k={k_retriever})."); pdf_processed_successfully = True
            else: print("FEHLER: Keine Chunks aus PDF erstellt.")
        else: print("FEHLER: Keine Seiten im PDF gefunden.")
    except ImportError as e: print(f"FEHLER: Benötigte Bibliothek fehlt: {e}")
    except Exception as e: print(f"FEHLER bei PDF-Verarbeitung/Vektorisierung: {e}")
if not pdf_processed_successfully: print("WARNUNG: PDF konnte nicht verarbeitet werden. RAG-Tool ist nicht verfügbar.")


# =============================================================================
# 3. Definition der Tools (Angepasst für globales LLM)
# =============================================================================

run_history = [] # Für Stats & Tracking

# --- Helper-Funktion für die Antwortgenerierung (nutzt globales llm) ---
def generate_answer_from_context(query: str, retrieved_docs: List[Any]) -> str:
    """Generiert eine Antwort basierend auf Kontext, nutzt globales llm."""
    global run_history, llm # Zugriff auf globales llm Objekt

    context_string = "\n\n".join([f"Auszug Seite {doc.metadata.get('page', '?')}:\n{doc.page_content}" for doc in retrieved_docs])
    prompt_template = """
Basierend AUSSCHLIESSLICH auf dem folgenden Kontext, beantworte die Frage des Benutzers.
Sei präzise und antworte direkt auf die Frage. Wenn der Kontext die Antwort nicht enthält,
gib an, dass die Information im bereitgestellten Kontext nicht gefunden wurde.
Gib keine Seitenzahlen an, es sei denn, sie sind Teil der Antwort.

Kontext:
{context}

Frage: {query}

Antwort:
"""
    prompt = prompt_template.format(context=context_string, query=query)
    # print(f" [Token Simulation: RAG LLM Input ~{len(prompt)//4} tokens]")

    if llm: # Prüfe, ob das globale llm Objekt initialisiert wurde
        try:
            print("    [RAG Tool] Generiere Antwort mit Ollama LLM...")
            llm_response = llm.invoke(prompt)
            # print(f" [Token Simulation: RAG LLM Output ~{len(llm_response)//4} tokens]")
            run_history[-1]["rag_step"] = "Antwort generiert (LLM)"
            return llm_response.strip()
        except Exception as e:
            print(f"FEHLER beim Aufruf des Ollama LLM für RAG: {e}")
            run_history[-1]["rag_step"] = f"Antwortgenerierung (LLM) fehlgeschlagen: {e}"
            return f"(Fehler bei LLM-Generierung. Kontext gefunden, aber Antwort konnte nicht formuliert werden: {e})"
    else:
        # --- Fallback zur Simulation, wenn LLM nicht verfügbar ---
        print("    [RAG Tool] LLM nicht verfügbar. Simuliere Antwortgenerierung...")
        # (Simulationslogik wie zuvor)
        query_lower = query.lower(); context_lower = context_string.lower()
        simulated_answer = f"(Simulierte Antwort, da LLM nicht verfügbar) Zu '{query}':\n"
        found_info = False
        if "schutzgebiet" in query_lower and "schutzgebiet" in context_lower: simulated_answer += "- In Schutzgebieten gelten oft besondere Regeln.\n"; found_info = True
        if "energieeffizienz" in query_lower and "anforderungen" in context_lower: simulated_answer += "- Anforderungen an Energieeffizienz betreffen Dämmung etc.\n"; found_info = True
        if not found_info: simulated_answer += "- Kontext gefunden, aber spezifische Antwort konnte nicht simuliert werden.\n"
        run_history[-1]["rag_step"] = "Antwort generiert (Simuliert)"
        return simulated_answer

# --- Tool 1: RAG für Informationssuche (ANGEPASST) ---
def rag_tool(query: str) -> str:
    """Sucht in Dokumenten und generiert eine direkte Antwort via LLM (oder Simulation)."""
    global run_history # Nutzt globales llm implizit via generate_answer_from_context
    run_history.append({"tool": "rag_tool", "query": query, "timestamp": datetime.datetime.now(), "rag_step": "Start"})

    if not retriever:
        run_history[-1]["result"] = "Fehler: Retriever nicht verfügbar"; run_history[-1]["rag_step"] = "Fehler (Kein Retriever)"
        return "FEHLER: RAG-Retriever ist nicht verfügbar (PDF nicht verarbeitet)."
    try:
        print(f"    [RAG Tool] Suche relevante Dokumententeile für: '{query}'")
        docs = retriever.invoke(query)
        run_history[-1]["rag_step"] = f"Retrieval abgeschlossen ({len(docs)} Chunks gefunden)"

        if not docs:
            run_history[-1]["result"] = "Keine Infos gefunden"
            return f"Ich konnte leider keine spezifischen Informationen zu '{query}' in den hinterlegten Dokumenten finden."

        # Antwortgenerierung aufrufen (nutzt globales llm, falls verfügbar)
        final_answer = generate_answer_from_context(query, docs)
        run_history[-1]["result"] = "Antwort generiert" # Oder spezifischer aus generate_answer...
        return final_answer

    except Exception as e:
        run_history[-1]["result"] = f"Fehler: {e}"; run_history[-1]["rag_step"] = f"Fehler während RAG: {e}"
        return f"Ein Fehler ist bei der Suche oder Antwortgenerierung aufgetreten: {e}"

# --- Tool 2: Grundstücksflächen-Rechner ---
def berechne_grundstücksfläche(länge: float, breite: float) -> Union[float, None]:
    global run_history
    run_history.append({"tool": "berechne_grundstücksfläche", "länge": länge, "breite": breite, "timestamp": datetime.datetime.now()})
    if länge is not None and breite is not None and länge > 0 and breite > 0:
        fläche = länge * breite; print(f"    [Rechner Tool] Fläche berechnet: {fläche:.2f} qm"); run_history[-1]["result"] = fläche; return fläche
    else: print("    [Rechner Tool] Ungültige Eingabe."); run_history[-1]["result"] = "Fehler: Ungültige Eingabe"; return None

# --- Tool 3: LLM für Umformulierung (ANGEPASST, nutzt globales llm) ---
def llm_rephrasing_tool(text_to_rephrase: str) -> str:
    """Nutzt das globale Ollama LLM (falls verfügbar) oder simuliert Umformulierung."""
    global run_history, llm # Zugriff auf globales llm Objekt
    run_history.append({"tool": "llm_rephrasing_tool", "text": text_to_rephrase, "timestamp": datetime.datetime.now()})

    if llm: # Prüfe, ob das globale llm Objekt initialisiert wurde
        try:
            print("    [Rephrasing Tool] Formuliere um mit Ollama LLM...")
            prompt = f"Formuliere den folgenden Satz professioneller und klarer für einen offiziellen Bauantrag: '{text_to_rephrase}'"
            # print(f" [Token Simulation: Rephrasing LLM Input ~{len(prompt)//4} tokens]")
            response = llm.invoke(prompt)
            # print(f" [Token Simulation: Rephrasing LLM Output ~{len(response)//4} tokens]")
            result = f"Vorschlag (vom LLM {OLLAMA_MODEL_NAME}):\n'{response.strip()}'" # Modellnamen anzeigen
            run_history[-1]["result"] = "Erfolgreich (LLM)"
            return result
        except Exception as e:
            print(f"FEHLER beim Aufruf des Ollama LLM für Rephrasing: {e}")
            run_history[-1]["result"] = f"Fehler (LLM): {e}"
            # Fallback zur Simulation
            result = f"(Fehler bei LLM-Umformulierung. Simulation wird verwendet.)\n" + simulate_rephrasing(text_to_rephrase)
            return result
    else:
        # --- Fallback zur Simulation ---
        print("    [Rephrasing Tool] LLM nicht verfügbar. Simuliere Umformulierung...")
        result = simulate_rephrasing(text_to_rephrase)
        run_history[-1]["result"] = "Erfolgreich (Simuliert)"
        return result

def simulate_rephrasing(text_to_rephrase: str) -> str:
     """Helper für die Simulation der Umformulierung."""
     original_lower = text_to_rephrase.lower()
     if "wird gewerblich genutzt" in original_lower: return "(Simulation) Vorschläge: 'Dient gewerblichen Zwecken.', 'Gewerbliche Nutzung vorgesehen.'"
     elif "haus bauen" in original_lower: return "(Simulation) Vorschläge: 'Errichtung eines Einfamilienhauses.', 'Neubau Wohngebäude.'"
     else: return f"(Simulation) Bessere Formulierung für '{text_to_rephrase}': '[Professionellere Formulierung]'"

# --- Tool 4: Feld-Beschreibung holen --- 
def get_field_description(field_path: str) -> str:
    global run_history
    run_history.append({"tool": "get_field_description", "field_path": field_path, "timestamp": datetime.datetime.now()})
    descriptions = Bauantrag.get_all_field_descriptions(); description = descriptions.get(field_path)
    if description: result = f"Beschreibung für '{field_path}': {description}"
    else:
        parts = field_path.split('.'); parent_path = ".".join(parts[:-1]) if len(parts) > 1 else None
        parent_desc = descriptions.get(parent_path) if parent_path else None
        if parent_desc: result = f"Konnte '{field_path}' nicht direkt finden. Übergeordnetes Feld '{parent_path}': {parent_desc}"
        else: result = f"FEHLER: Feld '{field_path}' nicht im Formular gefunden."
    run_history[-1]["result"] = "Erfolgreich" if description else "Fehler"; return result

# --- Tool 5: Formularfeld aktualisieren ---
def update_form_field(state: Bauantrag, field_path: str, value: Any) -> tuple[Bauantrag, str]:
    global run_history
    run_history.append({"tool": "update_form_field", "field_path": field_path, "value": value, "timestamp": datetime.datetime.now()})
    try:
        keys = field_path.split('.'); target_obj = state
        for key in keys[:-1]:
            if hasattr(target_obj, key): target_obj = getattr(target_obj, key)
            else: raise AttributeError(f"Zwischenobjekt '{key}' nicht gefunden.")
        field_name = keys[-1]
        if not hasattr(target_obj, field_name): raise AttributeError(f"Feld '{field_name}' nicht gefunden.")
        model_fields = target_obj.model_fields; converted_value = value
        if field_name in model_fields:
             field_info = model_fields[field_name]; target_type = field_info.annotation
             if hasattr(target_type, '__origin__') and target_type.__origin__ is Union:
                  non_none_types = [t for t in target_type.__args__ if t is not type(None)]
                  if non_none_types: target_type = non_none_types[0]
             print(f"    [Update Tool] Konvertiere '{value}' (Typ: {type(value)}) -> Typ '{target_type}' für '{field_path}'...")
             try:
                 if target_type is bool:
                     if isinstance(value, str):
                         v_lower = value.strip().lower()
                         if v_lower in ['ja', 'yes', 'true', '1']: converted_value = True
                         elif v_lower in ['nein', 'no', 'false', '0']: converted_value = False
                         else: raise ValueError("Für Ja/Nein bitte 'ja' oder 'nein' eingeben.")
                     else: converted_value = bool(value)
                 elif target_type is int: converted_value = int(float(str(value).replace(',','.'))) # Komma ersetzen
                 elif target_type is float: converted_value = float(str(value).replace(',','.')) # Komma ersetzen
                 elif target_type is str: converted_value = str(value)
                 else: converted_value = value
             except (ValueError, TypeError) as conv_e:
                  msg = f"FEHLER: Konnte Wert '{value}' nicht in Typ '{target_type}' umwandeln: {conv_e}"
                  run_history[-1]["result"] = msg; return state, msg
        setattr(target_obj, field_name, converted_value)
        msg = f"Formularfeld '{field_path}' erfolgreich auf '{converted_value}' gesetzt."
        print(f"    [Update Tool] {msg}"); run_history[-1]["result"] = "Erfolgreich"; return state, msg
    except (AttributeError, IndexError, ValueError, TypeError) as e:
        msg = f"FEHLER beim Aktualisieren von '{field_path}': {e}"
        print(f"    [Update Tool] {msg}"); run_history[-1]["result"] = msg; return state, msg

# --- Tool 6: Formularstatus anzeigen ---
def get_form_status(state: Bauantrag) -> str:
    global run_history
    run_history.append({"tool": "get_form_status", "timestamp": datetime.datetime.now()})
    status = "Aktueller Formularstatus:\n"; missing_fields = []
    try:
        state_dict = state.model_dump(exclude_unset=False)
        def check_fields(data, prefix=""):
            nonlocal status
            if isinstance(data, dict):
                for key, value in data.items():
                    field_path = f"{prefix}{key}"
                    if isinstance(value, dict): status += f"- {field_path}: [Bereich]\n"; check_fields(value, prefix=f"{field_path}.")
                    else:
                        status_line = f"  - {field_path}: {'✓ Ausgefüllt' if value is not None else '○ Offen'}"
                        if value is not None: status_line += f" (Wert: {value})\n"
                        else: status_line += "\n"; missing_fields.append(field_path)
                        status += status_line
        check_fields(state_dict)
        if missing_fields: status += f"\nOffene Felder: {', '.join(missing_fields)}"
        else: status += "\nAlle Felder scheinen ausgefüllt."
        run_history[-1]["result"] = "Erfolgreich"; return status
    except Exception as e: msg = f"Fehler beim Abrufen des Status: {e}"; run_history[-1]["result"] = msg; return msg

# --- Tool 7: Formular speichern ---
def save_form_to_json(state: Bauantrag, filepath: str = "bauantrag_output.json") -> str:
    global run_history
    run_history.append({"tool": "save_form_to_json", "filepath": filepath, "timestamp": datetime.datetime.now()})
    try:
        json_output = state.model_dump_json(indent=2)
        with open(filepath, "w", encoding="utf-8") as f: f.write(json_output)
        msg = f"Formular erfolgreich in '{filepath}' gespeichert."
        print(f"    [Save Tool] {msg}"); run_history[-1]["result"] = "Erfolgreich"; return msg
    except (IOError, TypeError, ValidationError) as e:
        msg = f"FEHLER beim Speichern als JSON: {e}"
        print(f"    [Save Tool] {msg}"); run_history[-1]["result"] = msg; return msg

print("Tools definiert.")


# =============================================================================
# 4. Agenten-Logik & Kontextmanagement & Anwendungsfunktionalität
# =============================================================================
def main_interaction_loop():
    global run_history, LLM_AVAILABLE # Zugriff auf LLM Status
    print("\n" + "="*50 + f"\n Bauantrags-Assistent PoC (LLM: {OLLAMA_MODEL_NAME if LLM_AVAILABLE else 'Nicht verfügbar - Simulation'})" + "\n" + "="*50)
    print("Hallo! Ich beantworte Fragen basierend auf den Vorschriften, helfe bei Formulierungen,")
    print("berechne Flächen und fülle das Formular aus.")
    print("Befehle: 'hilfe [feldpfad]', 'setze [feldpfad] auf [wert]', 'status', 'speichern', 'exit'.")

    bauantrag_state = Bauantrag()
    run_history = []

    while True:
        try:
            user_input = input("\nIhre Frage oder Anweisung: ").strip()
            if not user_input: continue
            user_input_lower = user_input.lower()

            if user_input_lower == "exit": print("Assistent wird beendet."); break
            if user_input_lower == "speichern":
                response = save_form_to_json(bauantrag_state)
                print(f"Assistent: {response}\nAssistent wird beendet."); break

            response = ""
            # --- Tool Routing (Keyword-basiert) ---
            if user_input_lower == "status": response = get_form_status(bauantrag_state)
            elif user_input_lower.startswith("hilfe "):
                parts = user_input.split(" ", 1)
                if len(parts) == 2:
                    field_path = parts[1].strip()
                    if field_path in Bauantrag.get_all_field_descriptions(): response = get_field_description(field_path)
                    else: response = f"'{field_path}' ist kein gültiger Feldpfad."
                else: response = "Bitte Feld angeben (z.B. 'hilfe projekt.bauprojekt_art')."
            elif user_input_lower.startswith(("setze ", "set ")):
                 parts = user_input.split(" auf ", 1)
                 if len(parts) == 2:
                     field_part = parts[0].split(" ", 1)
                     if len(field_part) == 2:
                         field_path = field_part[1].strip(); value = parts[1].strip()
                         if field_path in Bauantrag.get_all_field_descriptions(): bauantrag_state, response = update_form_field(bauantrag_state, field_path, value)
                         else: response = f"'{field_path}' ist kein gültiger Feldpfad."
                     else: response = "Format: 'setze [feldpfad] auf [wert]'"
                 else: response = "Format: 'setze [feldpfad] auf [wert]'"
            elif "berechne" in user_input_lower and "fläche" in user_input_lower and "länge" in user_input_lower and "breite" in user_input_lower:
                 try:
                     parts = user_input_lower.split(); l_idx = parts.index("länge")+1; b_idx = parts.index("breite")+1
                     l = float(parts[l_idx].replace('m','').replace(',','.')); b = float(parts[b_idx].replace('m','').replace(',','.'))
                     fläche = berechne_grundstücksfläche(l, b)
                     if fläche is not None:
                         print(f"Assistent: Fläche ist {fläche:.2f} qm. Feld 'grundstueck.grundstueckflaeche_qm' wird aktualisiert.")
                         bauantrag_state, update_msg = update_form_field(bauantrag_state, "grundstueck.grundstueckflaeche_qm", fläche)
                         response = update_msg
                     else: response = "Konnte Fläche nicht berechnen."
                 except (ValueError, IndexError): response = "Konnte Länge/Breite nicht extrahieren."
            elif "ausdrücken" in user_input_lower or "umformulieren" in user_input_lower or "besser sagen" in user_input_lower:
                 text_to_rephrase = user_input # Vereinfacht
                 response = llm_rephrasing_tool(text_to_rephrase) # Nutzt jetzt globales llm oder Simulation
            else: # Fallback: RAG
                if pdf_processed_successfully: response = rag_tool(user_input) # Nutzt jetzt globales llm oder Simulation
                else: response = "Wissensdatenbank (PDF) nicht verfügbar."

            print(f"\nAssistent: {response}")

        except KeyboardInterrupt: print("\nAssistent wird durch Benutzer unterbrochen."); break
        except Exception as e: print(f"\nUnerwarteter Fehler: {e}"); break

    print("\n" + "="*50 + "\n Bauantrags-Assistent PoC - Ende\n" + "="*50)

    # =============================================================================
    # 5. Stats und Tracking (Ausgabe der History)
    # =============================================================================
    print("\n--- Ausgeführte Aktionen (Run History) ---")
    if run_history:
        for i, item in enumerate(run_history):
            print(f"{i+1}. Tool: {item.get('tool', 'N/A')}")
            params = {k: v for k, v in item.items() if k not in ['tool', 'timestamp', 'result', 'rag_step']}
            if params: print(f"   Parameter: {params}")
            if 'rag_step' in item: print(f"   RAG Schritt: {item.get('rag_step')}")
            print(f"   Ergebnis: {item.get('result', 'N/A')}")
            print(f"   Zeitstempel: {item.get('timestamp')}")
        print(f"\nLLM Status: {'Verfügbar (' + OLLAMA_MODEL_NAME + ')' if LLM_AVAILABLE else 'Nicht verfügbar / Simuliert'}")
        print("Token Verbrauch (Platzhalter): Genaue Zählung nicht implementiert.")
    else: print("Keine Aktionen protokolliert.")


Bibliotheken importiert.
Versuche, Ollama LLM zu initialisieren (Modell: llama3.2:latest)...


/tmp/ipykernel_407/2608367141.py:51: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model=OLLAMA_MODEL_NAME)


INFO: Ollama LLM 'llama3.2:latest' erfolgreich initialisiert und verbunden.
Pydantic Models definiert.
Verarbeite PDF: assets/NÖ  BO 2014, Fassung gekürzt.pdf
-> PDF geladen (23 Seiten).
-> Dokument in 209 Chunks aufgeteilt.
-> Lade Embedding-Modell: sentence-transformers/all-MiniLM-L6-v2...


/tmp/ipykernel_407/2608367141.py:133: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  print(f"-> Lade Embedding-Modell: {embedding_model_name}..."); embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name); print("-> Embedding-Modell geladen.")


-> Embedding-Modell geladen.
-> Erstelle FAISS Vektordatenbank...
-> Vektordatenbank und Retriever erstellt (k=3).
Tools definiert.


=============================================================================
6. UseCase-Beschreibung und Mermaid Diagramm (10P) - Platzhalter
=============================================================================
'''
--- UseCase-Beschreibung ---

**Ziel:** Entwicklung eines Conversational AI Assistenten als Proof of Concept (PoC),
der Benutzer beim Ausfüllen eines (vereinfachten) Bauantragsformulars unterstützt.

**Problem:** Das Ausfüllen von Bauanträgen ist oft komplex, erfordert spezifisches Wissen
über Vorschriften und kann für Laien überfordernd sein. Unklare Formulierungen oder
fehlende Informationen können zu Verzögerungen führen.

**Lösung (PoC):** Ein Chatbot, der:
1.  Fragen zu spezifischen Feldern beantwortet (mittels hinterlegter Beschreibungen).
2.  Informationen aus relevanten Dokumenten (hier: Bauvorschriften-PDF) mittels RAG bereitstellt.
3.  Hilfestellungen bei Formulierungen gibt (simuliert durch LLM).
4.  Einfache Berechnungen durchführt (hier: Grundstücksfläche).
5.  Den Benutzer durch das Formular führt, den Status anzeigt und Felder aktualisiert.
6.  Das Endergebnis als strukturierte JSON-Datei speichert.

**Technologie-Stack (PoC):** Python, LangChain, Pydantic, FAISS (lokal), Sentence-Transformers, PyMuPDF.

**Abgrenzung:** Dieser PoC implementiert keine komplexe Dialogführung oder echtes LLM-basiertes
Agenten-Reasoning für die Tool-Auswahl. Die Agenten-Logik ist regelbasiert (Keywords).
Es findet keine externe Datenbankanbindung (z.B. für Schutzgebiets-Prüfung) statt.

--- Mermaid Sequenzdiagramm (Beispiel) ---

```mermaid
sequenceDiagram
    participant User
    participant Assistant (Agent Loop)
    participant RAG Tool
    participant Calculator Tool
    participant Form Tools
    participant State (Bauantrag Object)

    User->>Assistant (Agent Loop): Frage zu Vorschriften (z.B. "Was gilt im Schutzgebiet?")
    Assistant (Agent Loop)->>RAG Tool: process_query("Was gilt im Schutzgebiet?")
    RAG Tool-->>Assistant (Agent Loop): Relevante Text-Chunks
    Assistant (Agent Loop)-->>User: Antwort basierend auf RAG

    User->>Assistant (Agent Loop): Fläche berechnen (z.B. "Berechne Fläche Länge 20m Breite 10m")
    Assistant (Agent Loop)->>Calculator Tool: calculate_area(20, 10)
    Calculator Tool-->>Assistant (Agent Loop): 200.0
    Assistant (Agent Loop)->>User: "Fläche ist 200qm. In Feld 'grundstueck.grundstueckflaeche_qm' eintragen?"
    User->>Assistant (Agent Loop): "ja" (oder simuliert)
    # Wichtig: Auch in den Interaktionen den Namen in Anführungszeichen verwenden!
    Assistant (Agent Loop)->>"Form Tools": update_field("grundstueck.grundstueckflaeche_qm", 200.0)
    "Form Tools"->>State (Bauantrag Object): Setze Wert
    State (Bauantrag Object)-->>"Form Tools": OK
    "Form Tools"-->>Assistant (Agent Loop): Erfolgsmeldung
    Assistant (Agent Loop)-->>User: "Feld aktualisiert."

    User->>Assistant (Agent Loop): "status"
    Assistant (Agent Loop)->>"Form Tools": get_status()
    "Form Tools"->>State (Bauantrag Object): Lese Werte
    State (Bauantrag Object)-->>"Form Tools": Aktuelle Werte
    "Form Tools"-->>Assistant (Agent Loop): Statusbericht
    Assistant (Agent Loop)-->>User: Aktueller Formularstatus

    User->>Assistant (Agent Loop): "speichern"
    Assistant (Agent Loop)->>"Form Tools": save_to_json()
    "Form Tools"->>State (Bauantrag Object): Lese Werte
    State (Bauantrag Object)-->>"Form Tools": Alle Werte
    "Form Tools"-->>Assistant (Agent Loop): Speicher-Bestätigung
    Assistant (Agent Loop)-->>User: "Gespeichert. Tschüss!"

```
'''


=============================================================================
7. Erfolgreiche / Weniger erfolgreiche Fälle (05P) - Pl
=============================================================================

'''
--- Erfahrungen: Erfolgreiche und weniger erfolgreiche Fälle ---

**Erfolgreiche Fälle (Beispiele):**
1.  **RAG-Anfrage zu spezifischem Begriff:**
    *   User: "Was gilt im Schutzgebiet?"
    *   Assistent: (Liefert relevante Sätze aus dem PDF, falls vorhanden) -> Funktioniert gut, wenn die Info explizit im PDF steht.
2.  **Flächenberechnungen:**
    *   User: "Erechne mir die Grundstücksfläche l=20m b=10m"
    *   Assistent: (Rechnet die Grundstücksfläche aus und printed sie in den Chat)

3.  **Statusabfrage:**
    *   User: "status"
    *   Assistent: (Listet alle Felder mit Werten oder als 'Offen' auf) -> Gibt guten Überblick.

**Weniger erfolgreiche Fälle / Herausforderungen (Beispiele):**
1.  **Vage RAG-Anfragen:**
    *   User: "Ist mein Bauvorhaben genehmigungsfähig?"
    *   Assistent: (Liefert ggf. allgemeine Textstellen, aber keine konkrete Ja/Nein-Antwort) -> RAG liefert nur Text, keine Bewertung. Erwartungshaltung des Users wird nicht erfüllt.
2.  **Feld-Aktualisierung mit korrekten Daten:**
    *   User: "setze projekt.gebaeudehoehe_geplant auf 9.5"
    *   Assistent: "Formularfeld 'projekt.gebaeudehoehe_geplant' erfolgreich auf '9.5' gesetzt." -> Klappt zuverlässig bei korrekter Syntax und Datentyp.
3.  **Fehlerhafte Feld-Aktualisierung (Syntax/Typ):**
    *   User: "setze Höhe auf neun meter"
    *   Assistent: "FEHLER: Konnte Wert 'neun meter' nicht in den erwarteten Typ '<class 'float'>' für Feld 'projekt.gebaeudehoehe_geplant' umwandeln..." -> Keyword-basierte Logik ist nicht robust genug für natürliche Spracheingabe bei Werten. Typkonvertierung scheitert.
    *   User: "setze projekt.hoehe auf 9.5"
    *   Assistent: "'projekt.hoehe' scheint kein gültiger Feldpfad zu sein..." -> Tippfehler im Feldnamen werden nicht erkannt.
4.  **Kontextverständnis bei Folgefragen:**
    *   User: "Was gilt für Schutzgebiete?"
    *   Assistent: (Liefert Infos zu Schutzgebieten)
    *   User: "Und wie hoch darf ich dort bauen?"
    *   Assistent: (Startet neue RAG-Suche für "wie hoch darf ich dort bauen?", ohne den Kontext "Schutzgebiet" sicher beizubehalten) -> Fehlendes Dialoggedächtnis.
5.  **Grenzen der Keyword-Erkennung:**
    *   User: "Ich möchte die Fläche wissen, Länge ist 10, Breite 20."
    *   Assistent: (Erkennt ggf. nicht das Rechner-Tool, da "berechne" fehlt, und führt RAG durch) -> Agenten-Logik zu simpel.
'''

=============================================================================
Anwendung starten
=============================================================================

In [ ]:
if __name__ == "__main__":
    main_interaction_loop()


 Bauantrags-Assistent PoC (LLM: llama3.2:latest)
Hallo! Ich beantworte Fragen basierend auf den Vorschriften, helfe bei Formulierungen,
berechne Flächen und fülle das Formular aus.
Befehle: 'hilfe [feldpfad]', 'setze [feldpfad] auf [wert]', 'status', 'speichern', 'exit'.
    [RAG Tool] Suche relevante Dokumententeile für: 'Was gilt im Schutzgebiet?'
    [RAG Tool] Generiere Antwort mit Ollama LLM...
